In [9]:
# mount the drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
cd /content/drive/MyDrive/Colab Notebooks/

/content/drive/MyDrive/Colab Notebooks


In [3]:
from transformers import CLIPProcessor, CLIPModel
import torch
from PIL import Image

In [4]:
model_name = "openai/clip-vit-large-patch14"

model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [26]:
def get_text_embedding(text):
    # Added truncation and max_length to handle long text inputs
    inputs = processor(text=[text], return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)

    with torch.no_grad():
        outputs = model.get_text_features(**inputs)
        # Extract the tensor from the output object
        emb = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs

    # Normalize the embedding
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().cpu()

In [27]:
def get_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
        # Extract the tensor from the output object
        emb = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs

    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().cpu()

In [16]:
import os
def list_image_files(directory):
    image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp']
    image_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if any(file.lower().endswith(ext) for ext in image_extensions):
                image_files.append(os.path.join(root, file))
    return image_files

drive_path = '/content/drive/MyDrive/Colab Notebooks/images/'
all_image_files = list_image_files(drive_path)

if all_image_files:
    print("Found image files:")
    for img_file in all_image_files:
        print(img_file)
else:
    print("No image files found in Google Drive.")

Found image files:
/content/drive/MyDrive/Colab Notebooks/images/3653_IM-1815-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3660_IM-1820-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3663_IM-1822-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3679_IM-1831-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3694_IM-1845-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3677_IM-1830-2001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3684_IM-1836-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/367_IM-1826-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3675_IM-1829-0001-0001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3699_IM-1846-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3697_IM-1846-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3692_IM-1843-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3680_IM-1832-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/im

In [17]:
if all_image_files:
    image_to_use = all_image_files[0]  # Using the first found image as an example
    print(f"Using image: {image_to_use}")
    image_embedding = get_image_embedding(image_to_use)
    print(f"Length of image embedding: {len(image_embedding)}")
else:
    print("No image files found to process.")

Using image: /content/drive/MyDrive/Colab Notebooks/images/3653_IM-1815-1001.dcm.png
Length of image embedding: 768


In [8]:
UPSTASH_DB_URL = "https://enhanced-rhino-63976-eu1-vector.upstash.io"
UPSTASH_TOKEN = 'ABgFMGVuaGFuY2VkLXJoaW5vLTYzOTc2LWV1MWFkbWluTjJWbVpqTTRNV010TkRnd015MDBObUV6TFRreU1qRXRaV1V5WVdVM1pqZGlPV0pr'

In [21]:
import json
data = None
with open ('./impression_and_findings.json','r') as f:
  data = json.load(f)
print(data)

[{'id': 1, 'text': 'Findings: The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no  of a pleural effusion. There is no evidence of pneumothora Impression: Normal chest', 'image': '1_IM-0001-4001.dcm.png'}, {'id': 2, 'text': 'Findings: Borderline cardiomegaly. Midline sternotomy . Enlarged pulmonary arteries. Clear lungs. Inferior Impression: No acute pulmonary findings', 'image': '2_IM-0652-1001.dcm.png'}, {'id': 3, 'text': 'Findings: N/A Impression: No displaced rib fractures, pneumothora, or pleural effusion identified. Well-epanded and clear lungs. Mediastinal contour within normal limits. No acute cardiopulmonary abnormality identified', 'image': '3_IM-1384-1001.dcm.png'}, {'id': 4, 'text': 'Findings: There are diffuse bilateral interstitial and alveolar opacities consistent with chronic obstructive lung disease and bullous emphysema. There are irregular opacities in the left lung ape, that 

In [28]:
final_embeddings = []

for entry in data:
    # Get text embedding
    text_data = entry['text']
    text_embedding = get_text_embedding(text_data)

    # Get image embedding
    image_name = entry['image']
    image_path = os.path.join(drive_path, image_name) # Using the drive_path defined earlier
    image_embedding = get_image_embedding(image_path)

    # Calculate final embedding
    final_embedding = 0.6 * image_embedding + 0.4 * text_embedding
    final_embeddings.append(final_embedding)

    print(f"Entry ID: {entry['id']}")
    print(f"Final Embedding (first 5 elements): {final_embedding}")
    print(f"Length of Final Embedding: {len(final_embedding)}")
    print("----------------------------------------")


Entry ID: 1
Final Embedding (first 5 elements): tensor([-3.8153e-02,  5.0241e-02, -1.9060e-03,  4.1256e-02, -1.9593e-02,
         7.3943e-03, -1.9508e-02,  1.4860e-02,  8.6452e-03, -5.2417e-02,
        -4.0176e-02, -2.0063e-03, -5.6228e-02,  2.5183e-02, -5.7162e-03,
        -3.1825e-03, -1.7974e-02, -1.7181e-02,  3.7037e-02,  1.1774e-03,
         1.4879e-02, -5.5022e-03, -2.2692e-02,  2.7712e-02, -3.1209e-02,
        -4.3383e-03,  4.3336e-02, -2.4952e-02,  2.8879e-02, -7.7364e-03,
         1.3996e-02, -4.0561e-02, -1.9762e-02, -1.5230e-03,  8.4536e-03,
        -1.2055e-03,  3.7802e-02,  2.4812e-03,  6.1279e-04,  8.9022e-04,
         2.6845e-02, -1.4062e-02, -2.1807e-03,  8.6980e-03,  8.4501e-03,
         3.9574e-03, -1.0943e-02, -1.6952e-02, -2.7565e-02, -3.8480e-02,
        -3.0549e-02,  3.2063e-02, -1.9803e-02,  2.2484e-02,  2.2156e-02,
        -2.7237e-02, -1.0229e-02, -9.7889e-03,  2.9074e-02,  7.3005e-03,
         3.7083e-02, -8.6899e-03, -1.9664e-02,  3.8439e-02,  2.8719e-02,
   

KeyboardInterrupt: 